In [ ]:
import mhn
import sys
sys.path.append("../MetMHN")
import pandas as pd
import numpy as np
import jax.numpy as jnp

from metmhn.regularized_optimization import score
from metmhn.model import MetMHN


log_theta = pd.read_csv("../metmhn-analyses/2024-07-A-trr-extension/data/luad_g14_0005.csv", index_col=0)
obs1 = log_theta.iloc[0, :]
obs2 = log_theta.iloc[1, :]

log_theta.drop(index=[0, 1], inplace=True)
log_theta = log_theta.to_numpy()
log_theta[-1, -1] = 0 # set seeding base rate to 1 for now

metmhn_model = MetMHN(log_theta, obs1, obs2)

def metmhn_to_multimhn(log_theta, obs1, obs2):

    seeding_log_theta = log_theta.copy()
    seeding_log_theta[:-1, -1] = 0
    seeding_log_theta[-1, :-1] = obs1[:-1]
    omhn_seeding = mhn.model.oMHN(np.vstack([seeding_log_theta, np.hstack([log_theta[-1, :-1], 0])]))

    pt_log_theta = log_theta.copy()
    pt_log_theta[:- 1, -1] = 0
    omhn_pt = mhn.model.oMHN(np.vstack([pt_log_theta, obs1]))

    omhn_mt = mhn.model.oMHN(np.vstack([log_theta, obs2]))

    return omhn_seeding, omhn_pt, omhn_mt

omhn_seeding, omhn_pt, omhn_mt = metmhn_to_multimhn(log_theta, obs1, obs2)

In [ ]:
events = pd.read_csv("../metmhn-analyses/2024-07-A-trr-extension/data/G14_LUAD_Events.csv", index_col=0)
pat = events[events["paired"] == True].iloc[10].to_numpy(dtype=np.int32)[:-2]
pat

In [ ]:
# Obsorder
# 0: PT = MT 
# 1: MT > PT
# 2: PT < MT

# Metstatus
# 0: absent
# 1: present
# 2: isMetastasis
# 3: paired

np.exp(score(
        log_theta=jnp.array(log_theta),
        log_d_p=obs1.to_numpy(),
        log_d_m=obs2.to_numpy(),
        dat=np.hstack([pat, [0, 3]]).reshape(1, -1),
        perc_met=1))

This works

In [ ]:
# find possible split states
import itertools

final_pt_state = pat[::2]
final_mt_state = np.hstack([pat[1::2], pat[-1]])

i_seeding = log_theta.shape[1] - 1
pt_events = np.nonzero(final_pt_state)[0]
mt_events = np.nonzero(final_mt_state)[0]

powerset = lambda s: itertools.chain.from_iterable(itertools.combinations(list(s), r) for r in range(len(s)+1))
pt_split_distribution = np.zeros(1 << len(pt_events), dtype=float)
mt_split_distribution = np.zeros(1 << len(mt_events), dtype=float)
    
split_state = np.zeros_like(final_pt_state)

final_probability = 0

# iterate over all possible split_states aka subsets of shared events
for shared_events in powerset(np.nonzero((final_pt_state & final_mt_state)[:-1])[0]):

    shared_events = list(shared_events) + [i_seeding]
    
    split_state[list(shared_events)] = 1

    pt_split_distribution[int("".join(map(str, split_state[final_pt_state.astype(bool)][::-1])), 2)] = 1
    mt_split_distribution[int("".join(map(str, split_state[final_mt_state.astype(bool)][::-1])), 2)] = 1

    prob = omhn_seeding.compute_marginal_likelihood(split_state[:-1]) *\
        omhn_pt.compute_marginal_likelihood(final_pt_state, p0=pt_split_distribution) * \
        omhn_mt.compute_marginal_likelihood(final_mt_state, p0=mt_split_distribution)

    final_probability += prob

    pt_split_distribution.fill(0)
    mt_split_distribution.fill(0)
    split_state.fill(0)

print(final_probability)
